## INIT ALL CLASSES FOR FIXMATCH

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import (DataLoader, TensorDataset, Dataset, ConcatDataset)
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from einops.layers.torch import Rearrange
from timm.layers.drop import DropPath
from torch import randperm

from torch.utils.tensorboard import SummaryWriter
from itertools import cycle
from typing import Tuple, List, Optional, Callable, Union, Sequence
from tqdm.auto import tqdm

class EarlyStopping:
    def __init__(self, 
                 patience: int = 5, 
                 min_delta: float = 0.0, 
                 path: str = "checkpoint.pt",
                 verbose: bool = False):
        self.patience  = patience
        self.min_delta = min_delta
        self.path      = path
        self.verbose   = verbose
        self.counter   = 0
        self.best_loss = torch.inf
        self.early_stop = False

    def __call__(self, val_loss: float, model: torch.nn.Module):
        # check if loss improved by at least min_delta
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.path)
            if self.verbose:
                print(f"Validation loss improved to {val_loss:.4f}. Saved model to {self.path}")
        else:
            self.counter += 1
            if self.verbose:
                print(f"No improvement in val loss for {self.counter}/{self.patience} epochs.")
            if self.counter >= self.patience:
                self.early_stop = True

class ResnetBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int, dropout: float = 0.0): 
        # Adding group conv won't work well since there's no mixing mechanism in this
        # I need seperate Resnext block with a 1x1->3x3->1x1 not a 3x3->3x3
        super(ResnetBlock, self).__init__()
        
        self.dropout = dropout
        
        self.bn1 = nn.BatchNorm2d(in_channels, momentum = 0.01)
        self.LeakyReLU1 = nn.LeakyReLU(inplace = True, negative_slope = 0.01)
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size = 3, stride = stride, padding = 1, bias = False)
        self.bn2 = nn.BatchNorm2d(out_channels, momentum = 0.01)
        self.LeakyReLU2 = nn.LeakyReLU(inplace = True, negative_slope = 0.01)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size = 3, stride = 1, padding = 1, bias = False)
        self.InIsOut = in_channels == out_channels
        self.shortcutCompat = nn.Conv2d(in_channels, out_channels, kernel_size = 1, stride = stride, padding = 0, bias = False) if not self.InIsOut else nn.Identity() 
        
    def forward(self, x):
        if not self.InIsOut:
            x = self.LeakyReLU1(self.bn1(x))
        else:
            out = self.LeakyReLU1(self.bn1(x))

        
        out = self.LeakyReLU2(self.bn2(self.conv1(out if self.InIsOut else x)))
        if self.dropout > 0:
            out = F.dropout(out, self.dropout, training = self.training)
            
        out = self.conv2(out)
        
        return self.shortcutCompat(x) + out
    

class ConvNeXt(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int, dropout: float = 0.0, scaleInit: float = 1e-6):
        super(ConvNeXt, self).__init__()
        
        if in_channels != out_channels or stride != 1:
            self.reduction = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size = 2, stride = stride, bias = False),
                Rearrange("b c h w -> b h w c"),
                nn.LayerNorm(out_channels),
                Rearrange("b h w c -> b c h w"),
            )
        else:
            self.reduction = nn.Identity()

        self.compute    = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size = 7, stride = 1, padding = 3, groups = out_channels, bias = True),
            Rearrange("b c h w -> b h w c"),
            nn.LayerNorm(out_channels),
            nn.Linear(out_channels, out_channels * 4),
            nn.GELU(),
            nn.Linear(out_channels * 4, out_channels),
            Rearrange("b h w c -> b c h w"),
        )
        self.layerScale = nn.Parameter(scaleInit * torch.ones(out_channels), 
                                        requires_grad=True)
        self.dropPath   = DropPath(dropout)
            
    def forward(self, x: torch.Tensor):
        x = self.reduction(x)
        
        out = self.compute(x)
        out = self.layerScale.view(1, -1, 1, 1) * out
        
        out = self.dropPath(out)
        
        return x + out


class BlockStack(nn.Module):
    def __init__(self, blockType: nn.Module, in_channels: int, out_channels: int, stride: int, dropout: float, numBlock: int):
        super(BlockStack, self).__init__() 
        self.group = self.make_group(blockType, in_channels, out_channels, stride, dropout, numBlock)
        
    def make_group(self, blockType: nn.Module, in_channels: int, out_channels: int, stride: int, dropout: float, numBlock: int):
        layers = []
        for idx in range(numBlock):
            if idx == 0:
                layers.append(blockType(in_channels, out_channels, stride, dropout)) # This changes both resolution and channels
            else:
                layers.append(blockType(out_channels, out_channels, 1, dropout))
        
        return nn.Sequential(*layers)
    def forward(self, x):
        return self.group(x)

        

class WRN(nn.Module):
    def __init__(self, blockType: nn.Module, depth: int, widenFact: int, numClasses: int, patchSz: int = 0, dropout: float = 0.0):
        super(WRN, self).__init__()
        assert (depth - 4) % 6 == 0
        numBlock = (depth - 4) // 6

        channelDepth = [16, 16 * widenFact, 32 * widenFact, 64 * widenFact]
        strides = [1, 2, 2]

        self.patchSz = patchSz
        if patchSz != 0:
            self.stem = nn.Conv2d(3, channelDepth[0], kernel_size = patchSz, stride = patchSz) # The actual patchify layer
        else:
            self.stem = nn.Conv2d(3, channelDepth[0], kernel_size = 3, stride = 1) # Almost similar to patchify, but its overlapping kernel => no
        
        self.largeGroup = nn.ModuleList(
            [BlockStack(blockType, channelDepth[i], channelDepth[i + 1], strides[i], dropout, numBlock) for i in range(3)]
        )

        self.bn = nn.BatchNorm2d(channelDepth[-1], momentum = 0.01)
        self.LeakyReLU = nn.LeakyReLU(inplace = True, negative_slope = 0.01)
        self.fc = nn.Linear(channelDepth[-1], numClasses)

        
        for m in self.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                m.bias.data.zero_()
        
        
    def forward(self, x):
        if self.patchSz != 0 and not torch.jit.is_tracing():
            assert x.shape[-1] % self.patchSz == 0, f"Patch size is enabled but is not divisible by input shape. Redo the model"
        
        x = self.stem(x)
        for group in self.largeGroup:
            x = group(x)
        x = self.LeakyReLU(self.bn(x))
        x = F.adaptive_avg_pool2d(x, 1)
        x = torch.flatten(x, 1)
        return self.fc(x)

    def summary(self):
        
        total_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total_MB = total_params * 4 / (1024 ** 2)  # Assuming 32-bit float = 4 bytes
        print(f"Total Trainable Parameters: {total_params:,}")
        print(f"Approximate Model Size: {total_MB:.2f} MB")


def valSplit(train: tuple, split: float = 0.1):
    N = train[0].size(0)
    valLength   = int(split * train[0].shape[0])
    trainLength = train[0].shape[0] - valLength
    perm        = randperm(N, generator=torch.Generator().manual_seed(42))
    train_idx   = perm[:trainLength]
    val_idx     = perm[trainLength:]
    
    return (train[0][train_idx], train[1][train_idx]), (train[0][val_idx], train[1][val_idx])


class VariableTensorDataset(Dataset):
    def __init__(
        self,
        images: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        augments: Optional[List[Callable]] = None,
        release: Optional[Union[int, Sequence[int]]] = None,
    ):
        """
        images:   (N, C, H, W) tensor
        labels:   (N, ...) tensor or None
        augments: list of callables that map a Tensor->[Tensor]
        release:  which augment indices to return in __getitem__;
                  if None, uses self.aug_idx (single view)
                  if int or [i,j,...], returns tuple of views
        """
        super().__init__()
        if labels is not None:
            assert labels.shape[0] == images.shape[0], (
                f"Images/labels length mismatch: {images.shape[0]} vs {labels.shape[0]}"
            )

        assert isinstance(augments, (list, type(None))), "augments must be a list or None"
        self.images   = images
        self.labels   = labels
        self.augments = augments or []
        # the “default” single-view index
        self.aug_idx  = 0

        # normalize release into a list of ints, or None
        if release is None:
            self.release = None
        else:
            if isinstance(release, int):
                self.release = [release]
            else:
                # assume sequence of ints
                self.release = list(release)
            # sanity-check
            for i in self.release:
                assert 0 <= i < len(self.augments), f"release index {i} out of range"

    def set_augment(self, idx: int):
        """Change the single-view augment index (used when release is None)."""
        assert 0 <= idx < len(self.augments), "augment index out of range"
        self.aug_idx = idx

    def __len__(self):
        return self.images.size(0)

    def __getitem__(self, index):
        x = self.images[index]

        if self.augments:
            if self.release is not None:
                views = [self.augments[i](x) for i in self.release]
                x = tuple(views)
            else:
                x = self.augments[self.aug_idx](x)

        if self.labels is not None:
            return x, self.labels[index]
        else:
            return x, index
        
        
class VariableThresh(nn.Module):
    def __init__(self, unlabeledSz: int, numClasses: int, tau = .8):
        super().__init__()
        
        learningTracer = torch.full((unlabeledSz, ), -1, dtype = torch.long)
        self.register_buffer("learningTracer", learningTracer)
        
        classArray = torch.arange(numClasses).unsqueeze(0)
        self.register_buffer("classArray", classArray)
        
        self.N = unlabeledSz
        self.numClasses = numClasses
        self.tau = tau
        
    def forward(self, prob: torch.Tensor, indices):
        maxClass           = torch.argmax(prob, dim = 1, keepdim = True).expand(-1, self.numClasses)
        confs, pseudoLabel = prob.max(dim = 1)
        mask               = (confs >= self.tau)
        learnEffect        = (mask.unsqueeze(1) * (self.classArray == maxClass)).sum(dim = 0)

        confidentIndicies  = indices.to(self.learningTracer.device)[mask]
        self.learningTracer[confidentIndicies] = pseudoLabel[mask]
        
        
        unused = (self.learningTracer == -1).sum()
        if torch.max(learnEffect) < unused:
            denom = torch.max(torch.stack([
                torch.max(learnEffect),
                (self.N - torch.sum(learnEffect)).clone().detach()
            ]))
            beta = learnEffect / torch.clamp(denom, min = 1.0) # Per class (ratio of original threshold, also determines if the model is confident in that class)
        else:
            beta = learnEffect / torch.clamp(torch.max(learnEffect), min = 1.0) # Per class

        beta = self.project(beta)
        
        variMask = confs >= (beta[pseudoLabel] * self.tau)
        return variMask

    project = lambda self, x: x / (2 + x)

class DistributionAlignment(nn.Module):
    def __init__(self, labels: torch.tensor, numClasses: int, momentum: float):
        super(DistributionAlignment, self).__init__()
        
        counts = torch.bincount(labels)
        pEmperical = (counts.float() / labels.numel())
        
        self.register_buffer("pEmperical", pEmperical)
        self.register_buffer("pRunning", torch.zeros(numClasses))
        self.momentum = momentum
        
    def forward(self, q: torch.Tensor):
        pBatch = q.mean(dim = 0)
        self.pRunning = (
            self.momentum * self.pRunning 
            + (1 - self.momentum) * pBatch
        )
        
        labelTilde = q * (self.pEmperical / (self.pRunning + 1e-6)).unsqueeze(0)
        
        return labelTilde / labelTilde.sum(dim = 1, keepdim = True)

# Reading dataset and intialize cuda device

In [ ]:
device = torch.device('cuda')
torch.manual_seed(45)

useRatio = 1
unlabeled = torch.load("/kaggle/input/stl-10/unlabeled.pt"); unlabeled = unlabeled[: int(unlabeled.shape[0] * useRatio)].permute(0, 3, 1, 2)
train = torch.load("/kaggle/input/stl-10/train.pt"); trainX = train[0].to(torch.float32).permute(0, 3, 1, 2); trainY = train[1].long()
trainX = trainX / 255
unlabeled = unlabeled / 255
numClasses = len(torch.unique(trainY))


(trainX, trainY), (valX, valY) = valSplit((trainX, trainY), 0.15)

# Load the WRN model

In [ ]:
writer = SummaryWriter(log_dir = "/kaggle/working/FixMatchExperiment")
depth = 40; width = 2
model = WRN(ResnetBlock, depth, width, 10, 0, dropout = 0.25)

model.summary()
model.to(device)
writer.add_graph(model, trainX[:1].to(device))
writer.flush()

# Augment train and unlabeled dataset

In [ ]:
weakAugment = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(),
    T.ToTensor()
])

strongAugment = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(),
    T.RandAugment(num_ops = 3, magnitude = 10),
    T.ToTensor()
])


reAugmentApply = 2
trainDS     = VariableTensorDataset(trainX, trainY, augments = [weakAugment])
valDS       = VariableTensorDataset(valX,   valY,   augments = None)
unlabeledDS = VariableTensorDataset(unlabeled, augments = [weakAugment, ] + [strongAugment] * reAugmentApply, release = (0,) + tuple(range(1, 1 + reAugmentApply)))

trainSampleSz = len(trainDS); valSampleSz = len(valDS)

batchSize = 64; muy = 1.4
trainLoader     = DataLoader(trainDS, batch_size = batchSize, shuffle = True, num_workers = 4, pin_memory = True, persistent_workers = True)
unlabeledLoader = DataLoader(unlabeledDS, batch_size = int(muy * batchSize), shuffle = True, num_workers = 4, pin_memory = True, persistent_workers = True)
valLoader       = DataLoader(valDS, batch_size = 32, shuffle = True, num_workers = 4, pin_memory = True, persistent_workers = True)

## TRAINING TIME

In [ ]:
initLR = 1e-3; epochs = 580; tau = 0.9; l1 = 1e-5; l2 = 1e-5; 
    
optimizer             = optim.SGD(model.parameters(), lr = 1e-3, momentum = 0.9, nesterov = True)
scheduler             = CosineAnnealingLR(optimizer = optimizer, T_max = 20, eta_min = 0)
supervisedCriterion   = nn.CrossEntropyLoss(label_smoothing = 0.1)
unsupervisedCriterion = nn.CrossEntropyLoss(reduction = 'none')
alignment             = DistributionAlignment(trainY, numClasses = numClasses, momentum = 0.999).to(device)
thresh                = VariableThresh(len(unlabeledDS), numClasses, tau).to(device)
earlystop             = EarlyStopping(50, 0.00000001, path = f"/kaggle/working/Resnet_{depth}_{width}.pt", verbose = True)


pbar = tqdm(range(epochs), desc="Training Epochs")
for epoch in pbar:
    model.train()
    
    supervisedCost = 0
    consistencyCost = 0
    totalCost = 0
    trainCount = 0
    counter = 0
    for (xBatch, yBatch), ((unlabeledWeak, *unlabeledStrongList), indices) in zip(trainLoader, cycle(unlabeledLoader)):
        xBatch = xBatch.to(device, non_blocking = True)
        yBatch = yBatch.to(device, non_blocking = True)
        optimizer.zero_grad()

        logits         = model(xBatch.to(device))
        supervisedLoss = supervisedCriterion(logits, yBatch).mean()
        distribution   = torch.softmax(logits, dim = 1)
        trainCount     += (torch.argmax(distribution, dim = 1) == yBatch).sum().item(); counter += yBatch.shape[0]

        with torch.no_grad():
            unlabeledWeak = unlabeledWeak.to(device, non_blocking = True)
            wLogits       = model(unlabeledWeak)
            qWeak         = torch.softmax(wLogits, dim = 1)
            pseudoLabel   = alignment(qWeak)
            mask          = thresh(pseudoLabel, indices)
            # confs, pseudoLabel = qWeak.max(dim = 1)
            # pseudoLabel        = pseudoLabel.detach()
            # mask               = (confs >= tau).float()
            

        unsupervisedLosses = 0.0
        for unlabeledStrong in unlabeledStrongList:
            unlabeledStrong    = unlabeledStrong.to(device, non_blocking = True)
            sLogits            = model(unlabeledStrong)
            logProb            = F.log_softmax(sLogits, dim=1)
            loss               = F.kl_div(logProb, pseudoLabel, reduction="none").sum(dim=1)
            scalarLoss         = (mask * loss).mean()
            unsupervisedLosses += scalarLoss
        
        consistencyLoss = unsupervisedLosses / reAugmentApply

        weightParams = [p for n, p in model.named_parameters()
                        if p.requires_grad and "weight" in n]
        l1Norm = sum(p.abs().sum() for p in weightParams)
        l2Norm = sum(p.pow(2.0).sum() for p in weightParams)
        
        loss = supervisedLoss \
                + consistencyLoss \
                + l1Norm * l1 \
                + l2Norm * l2
        loss.backward()
        optimizer.step()


        supervisedCost  += supervisedLoss.item()
        consistencyCost += consistencyLoss.item()
        totalCost       += loss.item()
    
    trainSupLossTotal    = supervisedCost / len(trainLoader)
    consistencyLossTotal = consistencyCost / len(trainLoader)
    totalLoss            = totalCost / len(trainLoader)
    trainAcc             = trainCount / counter

    model.eval()
    runningLoss = 0.0; valCount = 0; counter = 0
    with torch.no_grad():
        for xBatch, yBatch in valLoader:
            xBatch = xBatch.to(device); yBatch = yBatch.to(device)

            outputs      = model(xBatch)
            loss         = supervisedCriterion(outputs, yBatch)
            distribution = torch.softmax(outputs, dim = 1)

            valCount    += (torch.argmax(distribution, dim = 1) == yBatch).sum().item()
            counter     += yBatch.shape[0]
            runningLoss += loss.item()

    valLossTotal = runningLoss / len(valLoader)
    valAcc = valCount / counter

    scheduler.step()
    currentLr = optimizer.param_groups[0]['lr']
    
    used     = torch.cuda.memory_allocated()  / 2**20
    reserved = torch.cuda.memory_reserved()   / 2**20

    
    tqdm.write(f"Epoch: {epoch + 1}, Supervised Loss: {trainSupLossTotal:.4f}, Consistency Loss: {consistencyLossTotal:.4f}, Loss: {totalLoss:.4f}, Train Accuracy: {100 * trainAcc:.2f}%, Val loss: {valLossTotal:.4f}, Val Acc: {100 * valAcc:.2f}%")
    pbar.set_postfix({
        "Supervised Loss": f"{trainSupLossTotal:.4f}",
        "Consistency Loss": f"{consistencyLossTotal:.4f}",
        "Loss": f"{totalLoss:.4f}",
        "Val Loss": f"{valLossTotal:.4f}"
    })  
    writer.add_scalar("Loss/Supervised",     trainSupLossTotal,    epoch + 1)
    writer.add_scalar("Loss/Consistency",    consistencyLossTotal, epoch + 1)
    writer.add_scalar("Loss/Total",          totalLoss,            epoch + 1)
    writer.add_scalar("Accuracy/Train",      100 * trainAcc,       epoch + 1)
    writer.add_scalar("Loss/Validation",     valLossTotal,         epoch + 1)
    writer.add_scalar("Accuracy/Validation", 100 * valAcc,         epoch + 1)
    writer.add_scalar("Misc/Lr",             currentLr,            epoch + 1)
    writer.add_scalar("Misc/GPU-used",       used,                 epoch + 1)
    writer.add_scalar("Misc/GPU-reserved",   reserved,             epoch + 1)
    writer.flush()
    
    earlystop(valLossTotal, model)
    if earlystop.early_stop:
        print(f"STOPPED AT EPOCH {epoch}")
        break